# Reto 1: descubre la clave de 8 dígitos dado un hash específico

## Información del problema

1. **Hash:** ef797c8118f02dfb649607dd5d3f8c7623048c9c063d532cc95c5ed7a898a64f
2. **Clave real:** 12345678

In [1]:
import hashlib

In [2]:
"""
    La siguiente función se encarga de probar por fuerza bruta diferentes claves numéricas,
    comparándolas contra un hash de referencia. Una vez que encuentra una clave numérica cuyo
    hash coincide con el hash ingresado, se detiene y muestra en pantalla la clave.

    Args:
        hash (str) : cadena de texto hexadecimal que representa al hash de referencia.
        lenght (int) : longitud de la cadena que se desea encontrar.
"""
def encontrar_clave_numerica(hash_value:str, length:int):

    # Bajo el supuesto de que la clave es solo numérica, con conocer la longitud de dígitos,
    # Podemos recorrer de forma creciente todas las permutaciones hasta el valor máximo posible
    # con esa cantidad de dígitos.
    num_permutations = 10**length
    for i in range(num_permutations):

        # Tomamos el número i, lo rellenamos con ceros a izquierda para cumplir la longitud dada,
        # lo convertimos en cadena, lo codificamos a un stream de bytes y aplicamos su hash.
        # Luego del hash, obtenemos la cadena en su representación hexadecimal

        string = hashlib.sha256(str(i).zfill(length).encode()).hexdigest()

        if hash_value == string:
            print(str(i).zfill(length))
            print(f"Total de hashes: {i}")
            break




In [3]:
# Ejecutamos el reto:
hash_value = "ef797c8118f02dfb649607dd5d3f8c7623048c9c063d532cc95c5ed7a898a64f"

encontrar_clave_numerica(hash_value=hash_value, length=8)

12345678
Total de hashes: 12345678


## Reflexiones:

### ¿Cuántas claves posibles?

Rta: basta con emplear el principio multiplicativo, cada posición tiene 10 posibilidades. El total de maneras es $10^8$, ya que la cadena tiene 8 posiciones.

### ¿Por qué no puede despejarse la clave?

Rta: porque las funciones hash no tienen inversa, es decir, no hay forma de deshacer el resultado de un hash para conocer qué dato lo generó.

### ¿Cuánto tarda una PC?

Rta: en mi máquina, tardó 51 s para realizar 12345678. Por lo tanto, la tasa de hash/segundo es de 242_072.11 hashes por segundo.


### ¿Existe otra clave con el mismo hash?

Para el conjunto de las $10^8$ claves, es poco probable. Sin embargo, teóricamente puede existir una cadena de longitud diferente de 8 que produzca el mismo hash, ya que sha-256 tiene un espacio de salida finito de $2^{256}$ valores.

# Reto 2: construye un árbol Merkle

In [4]:
"""
    Clase que se usa para representar un nodo del árbol.
    Se presupone que el árbol será binario, por lo que
    un nodo tendrá hijos izquierdo y derecho.
"""
class Nodo:
    def __init__(self, izquierdo, derecho, valor_hash, contenido, esta_copiado = False):
        self.izquierdo = izquierdo
        self.derecho = derecho
        self.valor_hash = valor_hash
        self.contenido = contenido
        self.esta_copiado = esta_copiado

    @staticmethod
    def aplicar_hash(valor : str) -> str:
        return hashlib.sha256(valor.encode()).hexdigest()

    def generar_copia(self):
        return Nodo(self.izquierdo, self.derecho, self.valor_hash, self.contenido, True)

    def __str__(self):
        return "Nodo = " + str(self.valor_hash)

    
"""
    La siguiente clase representa un árbol de Merkle
"""
class ArbolMerkle:

    def __init__(self, valores: list[str]):
        self.raiz = None
        self._crear_arbol(valores=valores)
        

    def _crear_arbol(self, valores:list[str]):
        """
            En este método se preparan los valores, que originalmente son cadenas,
            transformándolos en objetos de la clase Nodo, aquí se obtienen las hojas.

            Args:
            valores (list[str]) : es la lista de cadenas que serán las hojas.
        """
        hojas = []

        for valor in valores:
            nodo = Nodo(
                izquierdo= None,
                derecho= None,
                valor_hash= Nodo.aplicar_hash(valor),
                contenido= valor
            )

            hojas.append(nodo)

        if len(hojas) % 2 == 1:
            # Si la cantidad de hojas es impar, entonces duplicamos el último nodo
            ultimo_nodo = hojas[-1]
            hojas.append(ultimo_nodo.generar_copia())

        # Una vez obtenida la lista de nodos, se deben colocar en forma de árbol binario
        # Esto se logra recursivamente
        self.raiz = self._crear_arbol_recursivamente(hojas)

    def _crear_arbol_recursivamente(self, lista_nodos):


        # Si hay cantidad impar de nodos, copiamos el último de la lista
        # Así aseguramos siempre que un nodo padre tenga dos hijos
        if len(lista_nodos) % 2 != 0:
            ultimo_nodo = lista_nodos[-1]
            nodo_duplicado = ultimo_nodo.generar_copia()

            lista_nodos.append(nodo_duplicado)


        # Caso base: Si la lista tiene solo dos nodos, los convertimos en
        # hijos de un padre
        if len(lista_nodos) == 2:
            nodo_izquierdo = lista_nodos[0]
            nodo_derecho = lista_nodos[1]

            hash_padre = Nodo.aplicar_hash(nodo_izquierdo.valor_hash + nodo_derecho.valor_hash)

            contenido_padre = nodo_izquierdo.contenido + "+" + nodo_derecho.contenido

            return Nodo(
                izquierdo= nodo_izquierdo,
                derecho= nodo_derecho,
                valor_hash= hash_padre,
                contenido= contenido_padre
            )
        else:
            # Caso recursivo: construir el árbol por mitades

            mitad = len(lista_nodos) // 2 # Dividir lista en dos

            # Obtener dos sublistas
            nodos_izquierda = lista_nodos[:mitad] 
            nodos_derecha = lista_nodos[mitad:]


            # Subárbol izquierdo
            hijo_izquierdo = self._crear_arbol_recursivamente(nodos_izquierda)

            # Subárbol derecho
            hijo_derecho = self._crear_arbol_recursivamente(nodos_derecha)


            # Creamos el nodo padre que conecta los subárboles
            hash_padre = Nodo.aplicar_hash(hijo_izquierdo.valor_hash + hijo_derecho.valor_hash)

            contenido_padre = hijo_izquierdo.contenido + "+" + hijo_derecho.contenido

            # Devolvemos el nodo padre

            return Nodo(
                izquierdo=hijo_izquierdo,
                derecho=hijo_derecho,
                valor_hash=hash_padre,
                contenido=contenido_padre
            )

    def get_hash_raiz(self):
        return self.raiz.valor_hash

    def mostrar_arbol(self):
        """
            Este método se encarga de imprimir el árbol en pantalla
        """
        self._mostrar_arbol_recursivamente(self.raiz)

    def _mostrar_arbol_recursivamente(self, nodo:Nodo):

        # Caso base: si el nodo no existe, no muestra nada
        if nodo is None:
            return

        # Validar si el nodo es una hoja
        if nodo.izquierdo is None and nodo.derecho is None:
            print("Hoja (Input)")
        else:
            print("Hijo Izquierdo: ", nodo.izquierdo)
            print("Hijo Derecho: ", nodo.derecho)


        # Agregar información adicional en caso de que el nodo sea un duplicado de otro
        if nodo.esta_copiado:
            print("(Duplicado)")

        # Mostrar el valor hash y su contenido
        print("Valor Hash: ", nodo.valor_hash)
        print("Contenido del padre: ", nodo.contenido)
        print("\n")

        # Recorrer recursivamente el subárbol izquierdo
        self._mostrar_arbol_recursivamente(nodo.izquierdo)
        # Recorrer recursivamente el subárbol derecho
        self._mostrar_arbol_recursivamente(nodo.derecho)

    def obtener_prueba_inclusion(self, dato):
        """
            Genera la lista de los hashes hermanos al recorrer desde la hoja del dato hasta la ráiz, necesario para la verificación de la prueba de inclusión
        """

        prueba = []

        encontrado = self._buscar_dato_recursivamente(
                self.raiz,
                dato,
                prueba
            )

        if encontrado:
            return prueba

        return None

    def _buscar_dato_recursivamente(self, nodo: Nodo, dato, prueba):

        # No existe este nodo
        if nodo is None:
            return False

        # Validar si llegamos a una hoja
        if nodo.izquierdo is None and nodo.derecho is None:

            if nodo.contenido == dato:
                return True

            return False

        # Primero buscamos por la izquierda
        encontrado = self._buscar_dato_recursivamente(
            nodo.izquierdo,
            dato,
            prueba
        )

        if encontrado:

            # El dato estaba a la izquierda.
            # Guardamos el hash del hermano derecho.
            prueba.append(
                (nodo.derecho.valor_hash, "derecha")
            )

            return True

        # Si no estaba a la izquierda,
        # buscamos por la derecha
        encontrado = self._buscar_dato_recursivamente(
            nodo.derecho,
            dato,
            prueba
        )

        if encontrado:

            # El dato estaba a la derecha.
            # Guardamos el hash del hermano izquierdo.
            prueba.append(
                (nodo.izquierdo.valor_hash, "izquierda")
            )

            return True

        return False

            


In [5]:
# Poner a prueba los algoritmos
valores = ["Ana paga 150", "Luis paga 230", "Carlos paga 80", "Maria paga 95"]

# Instanciamos el árbol
arbol_merkle = ArbolMerkle(valores=valores)
# Mostramos el árbol
arbol_merkle.mostrar_arbol()

Hijo Izquierdo:  Nodo = 135b71cff480bda4120d4b5f619dead69c9da51f85ca55e7f4ce23f1e5b40c23
Hijo Derecho:  Nodo = f350e823c05810bcd858d68b85f3aae3312a027b66d272ce54ba3556ed328c25
Valor Hash:  3dee224f83ea08bc55ac2680bb5eced286ab775fca484acc3eafd757bbb71d93
Contenido del padre:  Ana paga 150+Luis paga 230+Carlos paga 80+Maria paga 95


Hijo Izquierdo:  Nodo = 7a18bfdcfe365d9598cb789513f9239a6cef11c4d2c2a1a6f32677e64777f97c
Hijo Derecho:  Nodo = 2f4c9726fa7bcc9b054a82470b582e2b504b8bd5aae1a4789eaefc76d5a774e1
Valor Hash:  135b71cff480bda4120d4b5f619dead69c9da51f85ca55e7f4ce23f1e5b40c23
Contenido del padre:  Ana paga 150+Luis paga 230


Hoja (Input)
Valor Hash:  7a18bfdcfe365d9598cb789513f9239a6cef11c4d2c2a1a6f32677e64777f97c
Contenido del padre:  Ana paga 150


Hoja (Input)
Valor Hash:  2f4c9726fa7bcc9b054a82470b582e2b504b8bd5aae1a4789eaefc76d5a774e1
Contenido del padre:  Luis paga 230


Hijo Izquierdo:  Nodo = 19b08e517d817544fc0baec4aa06a8c0ba9452be3b3e00c03af62c7c6da9fc46
Hijo Derecho:  

In [6]:
# Modificamos "Luis paga 999"
valores = ["Ana paga 150", "Luis paga 999", "Carlos paga 80", "Maria paga 95"]
# Instanciamos el árbol
arbol_merkle = ArbolMerkle(valores=valores)
# Mostramos el árbol
arbol_merkle.mostrar_arbol()

Hijo Izquierdo:  Nodo = 4c71e6c3bfc83df80daabda8343f0c8797d477d8f2fe3805ba662e0edcd4ad06
Hijo Derecho:  Nodo = f350e823c05810bcd858d68b85f3aae3312a027b66d272ce54ba3556ed328c25
Valor Hash:  90260bd30edf3480643ff8db6f640ae16753176a5b4ec7b57c21e711db96b610
Contenido del padre:  Ana paga 150+Luis paga 999+Carlos paga 80+Maria paga 95


Hijo Izquierdo:  Nodo = 7a18bfdcfe365d9598cb789513f9239a6cef11c4d2c2a1a6f32677e64777f97c
Hijo Derecho:  Nodo = 30dce18b91668fde50bd1ee19cb53c4f59080abfb8b372adc84d6ac35bcbe054
Valor Hash:  4c71e6c3bfc83df80daabda8343f0c8797d477d8f2fe3805ba662e0edcd4ad06
Contenido del padre:  Ana paga 150+Luis paga 999


Hoja (Input)
Valor Hash:  7a18bfdcfe365d9598cb789513f9239a6cef11c4d2c2a1a6f32677e64777f97c
Contenido del padre:  Ana paga 150


Hoja (Input)
Valor Hash:  30dce18b91668fde50bd1ee19cb53c4f59080abfb8b372adc84d6ac35bcbe054
Contenido del padre:  Luis paga 999


Hijo Izquierdo:  Nodo = 19b08e517d817544fc0baec4aa06a8c0ba9452be3b3e00c03af62c7c6da9fc46
Hijo Derecho:  

Se observa que, al cambiar "Luis paga 230" a "Luis paga 999", cambia su hash y el de los padres por encima de él hasta la ráiz.

# Reto 3: prueba de inclusión de Merkle

In [7]:
def verificar_prueba_inclusion(dato, prueba, raiz):
    """
        Aplicamos el algoritmo de clase:
        1. Hallamos hash del dato
        2. Hallamos hash de la concatenación del dato y el hermano.
        3. Obtenemos al padre.
        4. Realizamos el mismo procedimiento entre el padre y su hermano hasta llegar de regreso a la raíz
        5. Comparamos si el hash de la raíz y el hash calculado coinciden.
    """

    # Primero calculamos hash del dato (la hoja)
    hash_actual = Nodo.aplicar_hash(dato)

    # Recorremos los hashes hermanos de cada nivel
    for hash_hermano, posicion in prueba:

        if posicion == "izquierda":

            hash_actual = Nodo.aplicar_hash(
                hash_hermano + hash_actual
            )

        else:

            hash_actual = Nodo.aplicar_hash(
                hash_actual + hash_hermano
            )

    # Verificar si llegamos a la misma raíz
    return hash_actual == raiz

In [8]:
# Queremos probar la inclusión de "Luis paga 999"
valores = ["Ana paga 150", "Luis paga 999", "Carlos paga 80", "Maria paga 95"]
# Instanciamos el árbol
arbol_merkle = ArbolMerkle(valores=valores)
# Mostramos el árbol
arbol_merkle.mostrar_arbol()



Hijo Izquierdo:  Nodo = 4c71e6c3bfc83df80daabda8343f0c8797d477d8f2fe3805ba662e0edcd4ad06
Hijo Derecho:  Nodo = f350e823c05810bcd858d68b85f3aae3312a027b66d272ce54ba3556ed328c25
Valor Hash:  90260bd30edf3480643ff8db6f640ae16753176a5b4ec7b57c21e711db96b610
Contenido del padre:  Ana paga 150+Luis paga 999+Carlos paga 80+Maria paga 95


Hijo Izquierdo:  Nodo = 7a18bfdcfe365d9598cb789513f9239a6cef11c4d2c2a1a6f32677e64777f97c
Hijo Derecho:  Nodo = 30dce18b91668fde50bd1ee19cb53c4f59080abfb8b372adc84d6ac35bcbe054
Valor Hash:  4c71e6c3bfc83df80daabda8343f0c8797d477d8f2fe3805ba662e0edcd4ad06
Contenido del padre:  Ana paga 150+Luis paga 999


Hoja (Input)
Valor Hash:  7a18bfdcfe365d9598cb789513f9239a6cef11c4d2c2a1a6f32677e64777f97c
Contenido del padre:  Ana paga 150


Hoja (Input)
Valor Hash:  30dce18b91668fde50bd1ee19cb53c4f59080abfb8b372adc84d6ac35bcbe054
Contenido del padre:  Luis paga 999


Hijo Izquierdo:  Nodo = 19b08e517d817544fc0baec4aa06a8c0ba9452be3b3e00c03af62c7c6da9fc46
Hijo Derecho:  

In [9]:
# Obtener prueba de inclusión
prueba = arbol_merkle.obtener_prueba_inclusion(dato="Luis paga 999")
if prueba:
    print(verificar_prueba_inclusion(dato="Luis paga 999", prueba=prueba, raiz=arbol_merkle.get_hash_raiz()))
else:
    print("No se encontró el dato.")

True


# Reto 4: rediseñar el Árbol de Merkle, de manera tal que no duplique nodos

El objetivo es seguir la estrategia de "promoción", ahora no se duplica el nodo impar, sino que se "promociona" un nivel superior.

In [10]:
## Reutilizamos la misma clase Nodo
class Nodo:
    def __init__(self, izquierdo, derecho, valor_hash, contenido, esta_copiado = False):
        self.izquierdo = izquierdo
        self.derecho = derecho
        self.valor_hash = valor_hash
        self.contenido = contenido
        self.esta_copiado = esta_copiado

    @staticmethod
    def aplicar_hash(valor : str) -> str:
        return hashlib.sha256(valor.encode()).hexdigest()

    def generar_copia(self):
        return Nodo(self.izquierdo, self.derecho, self.valor_hash, self.contenido, True)

    def __str__(self):
        return "Nodo = " + str(self.valor_hash[:4])

# Construimos una versión rediseñada del árbol de Merkle  
"""
    La siguiente clase representa un árbol de Merkle
"""
class ArbolMerkle:

    def __init__(self, valores: list[str]):
        self.raiz = None
        if not valores:
            print("Ingresaste una lista vacía! No hay árbol que construir.")
            return
        
        self._crear_arbol(valores=valores)
        

    def _crear_arbol(self, valores:list[str]):
        """
            En este método se preparan los valores, que originalmente son cadenas,
            transformándolos en objetos de la clase Nodo, aquí se obtienen las hojas.

            Args:
            valores (list[str]) : es la lista de cadenas que serán las hojas.
        """
        hojas = []

        for valor in valores:
            nodo = Nodo(
                izquierdo= None,
                derecho= None,
                valor_hash= Nodo.aplicar_hash(valor),
                contenido= valor
            )

            hojas.append(nodo)

        # Una vez obtenida la lista de nodos, se deben colocar en forma de árbol binario
        # Esto se logra recursivamente
        self.raiz = self._crear_arbol_recursivamente(hojas)

    def _crear_arbol_recursivamente(self, lista_nodos):

        # Caso base: Si la lista tiene solo dos nodos, los convertimos en
        # hijos de un padre
        if len(lista_nodos) == 2:
            nodo_izquierdo = lista_nodos[0]
            nodo_derecho = lista_nodos[1]

            hash_padre = Nodo.aplicar_hash(nodo_izquierdo.valor_hash + nodo_derecho.valor_hash)

            contenido_padre = nodo_izquierdo.contenido + "+" + nodo_derecho.contenido

            return Nodo(
                izquierdo= nodo_izquierdo,
                derecho= nodo_derecho,
                valor_hash= hash_padre,
                contenido= contenido_padre
            )
        elif len(lista_nodos) == 1:
            # Segundo caso base, hemos aislado un nodo "impar"
            # Lo ascendemos, retornándolo como su propio padre
            return lista_nodos[0]
        else:
            # Caso recursivo: construir el árbol por mitades

            mitad = len(lista_nodos) // 2 # Dividir lista en dos

            # Obtener dos sublistas
            nodos_izquierda = lista_nodos[:mitad] 
            nodos_derecha = lista_nodos[mitad:]


            # Subárbol izquierdo
            hijo_izquierdo = self._crear_arbol_recursivamente(nodos_izquierda)

            # Subárbol derecho
            hijo_derecho = self._crear_arbol_recursivamente(nodos_derecha)


            # Creamos el nodo padre que conecta los subárboles
            hash_padre = Nodo.aplicar_hash(hijo_izquierdo.valor_hash + hijo_derecho.valor_hash)

            contenido_padre = hijo_izquierdo.contenido + "+" + hijo_derecho.contenido

            # Devolvemos el nodo padre

            return Nodo(
                izquierdo=hijo_izquierdo,
                derecho=hijo_derecho,
                valor_hash=hash_padre,
                contenido=contenido_padre
            )

    def get_hash_raiz(self):
        return self.raiz.valor_hash

    def mostrar_arbol(self):
        """
            Este método se encarga de imprimir el árbol en pantalla
        """
        self._mostrar_arbol_recursivamente(self.raiz)

    def _mostrar_arbol_recursivamente(self, nodo:Nodo):

        # Caso base: si el nodo no existe, no muestra nada
        if nodo is None:
            return

        # Mostrar el valor hash y su contenido, vamos a truncar el hash a solo 4 términos
        print("Valor Hash de este Nodo: ", nodo.valor_hash[:4], "...")
        print("Contenido del nodo: ", nodo.contenido)
        

        # Validar si el nodo es una hoja
        if nodo.izquierdo is None and nodo.derecho is None:
            print("Hoja (Input)")
            print("\n")
        else:
            print("Hijo Izquierdo: ", nodo.izquierdo)
            print("Hijo Derecho: ", nodo.derecho)
            print("\n")


        # Agregar información adicional en caso de que el nodo sea un duplicado de otro
        if nodo.esta_copiado:
            print("(Duplicado)")



        # Recorrer recursivamente el subárbol izquierdo
        self._mostrar_arbol_recursivamente(nodo.izquierdo)
        # Recorrer recursivamente el subárbol derecho
        self._mostrar_arbol_recursivamente(nodo.derecho)

    def obtener_prueba_inclusion(self, dato):
        """
            Genera la lista de los hashes hermanos al recorrer desde la hoja del dato hasta la ráiz, necesario para la verificación de la prueba de inclusión
        """

        prueba = []

        encontrado = self._buscar_dato_recursivamente(
                self.raiz,
                dato,
                prueba
            )

        if encontrado:
            return prueba

        return None

    def _buscar_dato_recursivamente(self, nodo: Nodo, dato, prueba):

        # No existe este nodo
        if nodo is None:
            return False

        # Validar si llegamos a una hoja (Caso Base)
        if nodo.izquierdo is None and nodo.derecho is None:

            if nodo.contenido == dato:
                return True

            return False

        # Primero buscamos por la izquierda
        encontrado = self._buscar_dato_recursivamente(
            nodo.izquierdo,
            dato,
            prueba
        )

        if encontrado:

            # El dato estaba a la izquierda.
            # Guardamos el hash del hermano derecho.
            prueba.append(
                (nodo.derecho.valor_hash, "derecha")
            )

            return True

        # Si no estaba a la izquierda,
        # buscamos por la derecha
        encontrado = self._buscar_dato_recursivamente(
            nodo.derecho,
            dato,
            prueba
        )

        if encontrado:

            # El dato estaba a la derecha.
            # Guardamos el hash del hermano izquierdo.
            prueba.append(
                (nodo.izquierdo.valor_hash, "izquierda")
            )

            return True

        return False

            

    

In [11]:
lista = ["T1", "T2", "T3", "T4", "T5"] #Cantidad impar de transacciones
arbol = ArbolMerkle(lista)
arbol.mostrar_arbol()
print("############# RAÍZ: ", arbol.get_hash_raiz())



Valor Hash de este Nodo:  4ccd ...
Contenido del nodo:  T1+T2+T3+T4+T5
Hijo Izquierdo:  Nodo = 87de
Hijo Derecho:  Nodo = 7aea


Valor Hash de este Nodo:  87de ...
Contenido del nodo:  T1+T2
Hijo Izquierdo:  Nodo = 1f93
Hijo Derecho:  Nodo = 0f61


Valor Hash de este Nodo:  1f93 ...
Contenido del nodo:  T1
Hoja (Input)


Valor Hash de este Nodo:  0f61 ...
Contenido del nodo:  T2
Hoja (Input)


Valor Hash de este Nodo:  7aea ...
Contenido del nodo:  T3+T4+T5
Hijo Izquierdo:  Nodo = 5dd6
Hijo Derecho:  Nodo = 7622


Valor Hash de este Nodo:  5dd6 ...
Contenido del nodo:  T3
Hoja (Input)


Valor Hash de este Nodo:  7622 ...
Contenido del nodo:  T4+T5
Hijo Izquierdo:  Nodo = 11ee
Hijo Derecho:  Nodo = 020d


Valor Hash de este Nodo:  11ee ...
Contenido del nodo:  T4
Hoja (Input)


Valor Hash de este Nodo:  020d ...
Contenido del nodo:  T5
Hoja (Input)


############# RAÍZ:  4ccd6a62c0d2399be8f277351b2512ebd4ee2b4d14d439df791069eb28d513a8


In [12]:
def verificar_prueba_inclusion(dato, prueba, raiz):
    """
        Aplicamos el algoritmo de clase:
        1. Hallamos hash del dato
        2. Hallamos hash de la concatenación del dato y el hermano.
        3. Obtenemos al padre.
        4. Realizamos el mismo procedimiento entre el padre y su hermano hasta llegar de regreso a la raíz
        5. Comparamos si el hash de la raíz y el hash calculado coinciden.
    """

    # Primero calculamos hash del dato (la hoja)
    hash_actual = Nodo.aplicar_hash(dato)

    # Recorremos los hashes hermanos de cada nivel
    for hash_hermano, posicion in prueba:

        if posicion == "izquierda":

            hash_actual = Nodo.aplicar_hash(
                hash_hermano + hash_actual
            )

        else:

            hash_actual = Nodo.aplicar_hash(
                hash_actual + hash_hermano
            )

    # Verificar si llegamos a la misma raíz
    return hash_actual == raiz

In [13]:
# Obtener prueba de inclusión
prueba = arbol.obtener_prueba_inclusion(dato="T5")
if prueba:
    print(verificar_prueba_inclusion(dato="T5", prueba=prueba, raiz=arbol.get_hash_raiz()))
else:
    print("No se encontró el dato.")

True
